# 23_v4 — Mini Transformer (Word2Vec v4 + Teacher Forcing + Window 32)

**Diferencia con 23_v3:**
1. **6 capas** (vs 3) — más capacidad
2. **LR Warmup** habilitado — estabiliza training con más capas
3. **Label Smoothing 0.1** — regularización extra
4. **Top-5 accuracy + Perplexity** — métricas complementarias
5. **Validation cada 2 epochs** — ~2× más rápido
6. **Colab-ready (T4 GPU)** — setup automático con Drive

Run with: conda activate tfenv  (local) o en Colab con T4 GPU

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


In [2]:
print('Loading iohadrubin/wikitext-103-raw-v1 (streaming)...')
ds = load_dataset('iohadrubin/wikitext-103-raw-v1', split='train', streaming=True)

Loading iohadrubin/wikitext-103-raw-v1 (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


dataset_infos.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

In [9]:
# Setup: añadir nlp_lib al path
import sys, os, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # En Colab clonamos/actualizamos repo
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/learning-ai'):
        !git clone https://github.com/eanorambuena/learning-ai.git /content/learning-ai
    else:
        !git -C /content/learning-ai pull
    !pip install datasets scikit-learn matplotlib -q
    sys.path.insert(0, '/content/learning-ai/notebooks/nlp')
    # Forzar recarga de nlp_lib si ya estaba importado
    import importlib
    if 'nlp_lib' in sys.modules:
        importlib.reload(sys.modules['nlp_lib'])
    print('Colab setup OK')
else:
    # Local: buscar notebooks/nlp/ en varios lugares
    _nb_dir = pathlib.Path(os.getcwd()).resolve()
    _candidates = [
        _nb_dir / 'notebooks' / 'nlp',
        _nb_dir,
    ]
    for _p in _candidates:
        if (_p / 'nlp_lib').is_dir():
            sys.path.insert(0, str(_p))
            break
    del _nb_dir, _p, _candidates

import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab setup OK
TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
# Hiperparámetros
WINDOW = 32             # contexto (32 palabras)
EMBED_DIM = 128         # dimensión embeddings (Word2Vec v4)
NUM_HEADS = 4
NUM_LAYERS = 6
FF_DIM = 128
DROPOUT = 0.2
BATCH_SIZE = 64
LR = 0.001
WARMUP_STEPS = 500
LABEL_SMOOTHING = 0.1  # regularización label smoothing
VALIDATION_FREQ = 2   # validar cada 2 epochs (acelera 2×)
PATIENCE = 10        # early stopping patience
MAX_CHARS = 5_000_000   # chars de wikitext-103 a cargar

In [11]:
from nlp_lib import Word2VecLoader

loader = Word2VecLoader(version='v4')

FileNotFoundError: [Errno 2] No such file or directory: '../myWord2Vec/v4/target_embeddings.npy'

In [ ]:
full_text = ''
for i, row in enumerate(ds):
    full_text += row['text'] + ' '
    if len(full_text) > MAX_CHARS:
        break
print(f'Loaded {i+1} rows, total chars: {len(full_text)}')

words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'-0123456789') for w in words]
words = [w for w in words if w in loader.text_vocab]
print(f'Total words used: {len(words)}')

In [ ]:
def create_sequences(words, window=WINDOW, step=1):
    X, y = [], []
    for i in range(0, len(words) - window, step):
        context = words[i:i + window]
        target = words[i+1:i + window + 1]   # teacher forcing: predecir en cada pos
        if all(w in loader.text_vocab for w in context + target):
            X.append([loader.text_vocab[w] for w in context])
            y.append([loader.text_vocab[w] for w in target])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)  # y: (N, window)

X, y = create_sequences(words, window=WINDOW)
print(f'Sequences created: {len(X)}, X shape: {X.shape}, y shape: {y.shape}')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)


In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angles = pos * angle_rates
    pe = np.zeros_like(angles)
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(pe[np.newaxis, :, :], tf.float32)

In [ ]:
# Diagrama de arquitectura del mini-Transformer (similar a "Attention is All You Need")

def draw_architecture_diagram():
    fig, ax = plt.subplots(1, 1, figsize=(10, 14))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 16)
    ax.axis('off')

    def draw_box(ax, x, y, w, h, text, color='lightblue'):
        rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                        facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, fontweight='bold')

    def draw_arrow(ax, x1, y1, x2, y2):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

    cx = 5  # center x
    col1 = 2.5  # left column
    w = 5
    h = 0.7

    # Input
    draw_box(ax, col1, 15.0, w, h, f'Input Tokens ({WINDOW} palabras)', 'lightgray')
    draw_arrow(ax, cx, 15.0, cx, 14.0)

    # Embedding + PosEncoding
    draw_box(ax, col1, 13.3, w, h, 'Embedding (Word2Vec) + PosEncoding', 'lightblue')
    draw_arrow(ax, cx, 13.3, cx + 2.5, 12.5)

    # x N blocks
    block_labels = ['Block 1', 'Block 2', 'Block 3']
    block_colors = ['#e8f5e9', '#c8e6c9', '#a5d6a7']
    bx = 2.5
    bw = 5
    bh = 2.8
    base_y = 9.6

    for i in range(3):
        y_off = base_y - i * 2.9
        color = block_colors[i]
        rect = mpatches.FancyBboxPatch((bx, y_off), bw, bh, boxstyle="round,pad=0.2",
                                       facecolor=color, edgecolor='green', linewidth=2, linestyle='--')
        ax.add_patch(rect)
        ax.text(bx + bw/2, y_off + bh - 0.3, block_labels[i],
                ha='center', va='center', fontsize=10, fontweight='bold', style='italic')

        # Sub-components inside block
        inner_w = 4
        inner_h = 0.55
        ix = bx + (bw - inner_w) / 2

        # Multi-Head Attention
        draw_box(ax, ix, y_off + 1.7, inner_w, inner_h,
                 'Multi-Head Self-Attention', '#fff9c4')
        # Add & Norm
        draw_box(ax, ix + 2, y_off + 1.1, inner_w - 2.5, 0.45,
                 'Add & Norm', '#ffccbc')
        # FFN
        draw_box(ax, ix, y_off + 0.5, inner_w, inner_h,
                 'Feed-Forward (Dense→ReLU→Dense)', '#b3e5fc')
        # Add & Norm
        draw_box(ax, ix + 2, y_off + 0.0, inner_w - 2.5, 0.45,
                 'Add & Norm', '#ffccbc')

        # Arrows between blocks
        if i < 2:
            y_next = base_y - (i+1) * 2.9 + bh
            draw_arrow(ax, cx, y_off, cx, y_next)

    draw_arrow(ax, cx, base_y - 2*2.9, cx, 3.3)

    # Pooling + Output
    draw_box(ax, col1, 2.6, w, h, 'Salida en todas las posiciones (Teacher Forcing)', 'lightblue')
    draw_arrow(ax, cx, 2.6, cx, 1.8)

    draw_box(ax, col1, 1.1, w, h, 'Dense → Softmax (vocab)', 'lightcoral')
    draw_arrow(ax, cx, 1.1, cx, 0.3)

    draw_box(ax, col1, -0.4, w + 0.0, h, 'Siguiente palabra', 'lightgreen')

    # Annotation: x N
    ax.text(cx + 0.5, 7.9, '× 3', fontsize=14, fontweight='bold', color='green')

    # Causal mask annotation
    ax.text(8.5, 8.0, '🔒 Causal mask', fontsize=9, fontstyle='italic', color='red',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.title('Mini-Transformer Architecture', fontsize=14, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()

draw_architecture_diagram()

In [ ]:
# ManualSelfAttention con Causal Mask
# mask[i][j] = 1 si j <= i (lower triangular), 0 en caso contrario
# scores = scores * mask + (1 - mask) * -1e9  (softmax de los valores enmascarados da ~0)

class CausalSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads=4, dropout_rate=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.Wq = layers.Dense(embed_dim, use_bias=False)
        self.Wk = layers.Dense(embed_dim, use_bias=False)
        self.Wv = layers.Dense(embed_dim, use_bias=False)
        self.Wo = layers.Dense(embed_dim, use_bias=False)
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, return_attention=False):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        Q = tf.reshape(Q, (B, T, self.num_heads, self.head_dim))
        K = tf.reshape(K, (B, T, self.num_heads, self.head_dim))
        V = tf.reshape(V, (B, T, self.num_heads, self.head_dim))
        Q = tf.transpose(Q, [0, 2, 1, 3])
        K = tf.transpose(K, [0, 2, 1, 3])
        V = tf.transpose(V, [0, 2, 1, 3])

        scale = tf.sqrt(tf.cast(self.head_dim, tf.float32))
        scores = tf.matmul(Q, K, transpose_b=True) / scale

        # Causal mask: lower triangular
        mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        mask = tf.reshape(mask, (1, 1, T, T))
        scores = scores * mask + (1 - mask) * -1e9

        att_weights = tf.nn.softmax(scores, axis=-1)
        att_weights = self.dropout(att_weights)
        context = tf.matmul(att_weights, V)
        context = tf.transpose(context, [0, 2, 1, 3])
        context = tf.reshape(context, (B, T, self.num_heads * self.head_dim))
        out = self.Wo(context)
        if return_attention:
            att_avg = tf.reduce_mean(att_weights, axis=1)
            return out, att_avg
        return out


# Transformer Block = Self-Attention + Add&Norm + FFN + Add&Norm
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=4, ff_dim=128, dropout_rate=0.1):
        super().__init__()
        self.attention = CausalSelfAttention(embed_dim, num_heads, dropout_rate)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
            layers.Dropout(dropout_rate),
        ])

    def call(self, x, return_attention=False):
        if return_attention:
            attn_out, attn_w = self.attention(x, return_attention=True)
        else:
            attn_out = self.attention(x)
            attn_w = None
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        if return_attention:
            return x, attn_w
        return x


# Mini Transformer = Embedding + PosEncoding + N Transformer Blocks + All Positions Output (Teacher Forcing)
class MiniTransformer(keras.Model):
    def __init__(self, embedding_layer, vocab_size, embed_dim, seq_len=5,
                 num_heads=4, num_layers=3, ff_dim=128):
        super().__init__()
        self.seq_len = seq_len
        self.embed_dim = embed_dim
        self.num_layers = num_layers

        self.embedding = embedding_layer
        self.pos_encoding = positional_encoding(seq_len, embed_dim)
        self.blocks = [TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)]
        self.out = layers.Dense(vocab_size)

    def call(self, inputs, return_attention=False):
        x = self.embedding(inputs)
        x = x + self.pos_encoding
        x = tf.nn.dropout(x, rate=0.1)

        attn_weights = None
        for i, block in enumerate(self.blocks):
            if return_attention and i == self.num_layers - 1:
                x, attn_w = block(x, return_attention=True)
                attn_weights = attn_w
            else:
                x = block(x)

        logits = self.out(x)  # (batch, seq_len, vocab_size) — teacher forcing: predecir en todas las pos

        if return_attention:
            return logits, attn_weights
        return logits

model = MiniTransformer(
    loader.embedding_layer, loader.vocab_size, loader.embedding_dim,
    seq_len=WINDOW, num_heads=NUM_HEADS, num_layers=NUM_LAYERS, ff_dim=FF_DIM
)

# Constant LR (fallback si warmup falla)

# LR Warmup (como en Attention is All You Need)
d_model = loader.embedding_dim
class WarmupSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return self.d_model ** (-0.5) * tf.minimum(step ** (-0.5), step * self.warmup_steps ** (-1.5))

optimizer = keras.optimizers.Adam(learning_rate=WarmupSchedule(d_model, warmup_steps))
model.compile(optimizer=optimizer, loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTHING), metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top_5_acc')])
model.build((None, WINDOW))

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(min(len(X_train), 2000)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('23_v4_best.weights.h5', monitor='val_accuracy', save_best_only=True, save_weights_only=True)
]

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=100, verbose=1, callbacks=callbacks, validation_freq=VALIDATION_FREQ)
model.summary()

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy (Mini Transformer, window={WINDOW}, {model.num_layers} layers): {acc:.3f}')

def predict_next_word(context_words, top_n=5):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    if len(context_ids) < WINDOW:
        context_ids = [0] * (WINDOW - len(context_ids)) + context_ids
    context_ids = np.array([context_ids[-WINDOW:]], dtype=np.int32)
    logits = model.predict(context_ids, verbose=0)  # (1, WINDOW, vocab_size)
    last_logits = logits[0, -1, :]                  # última posición = predicción de sig palabra
    probs = tf.nn.softmax(last_logits).numpy()
    top_indices = np.argsort(probs)[-top_n:][::-1]
    return [(loader.decode(idx), float(probs[idx])) for idx in top_indices]

sample_contexts = [
    ['london', 'bridge', 'river', 'city', 'center'],
    ['bank', 'of', 'england', 'is', 'located'],
    ['queen', 'of', 'england', 'lives', 'in']
]

for context in sample_contexts:
    preds = predict_next_word(context, top_n=5)
    if preds:
        print(f"Contexto {context} -> {preds}")

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title(f'Mini Transformer ({model.num_layers} layers, window={WINDOW})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Heatmap de atención causal (último bloque)
def plot_causal_attention(context_words, model):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    if len(context_ids) < WINDOW:
        context_ids = [0] * (WINDOW - len(context_ids)) + context_ids
    context_ids = np.array([context_ids[-WINDOW:]], dtype=np.int32)

    _, attn_avg = model(context_ids, return_attention=True)
    attn_matrix = attn_avg[0].numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    im = axes[0].imshow(attn_matrix, cmap='YlOrRd', vmin=0, vmax=attn_matrix.max())
    axes[0].set_title(f'Causal Self-Attention ({WINDOW}x{WINDOW})')
    axes[0].set_xlabel('Palabra atendida (j)')
    axes[0].set_ylabel('Palabra que atiende (i)')
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    attended = attn_matrix.sum(axis=0)
    attending = attn_matrix.sum(axis=1)
    positions = np.arange(WINDOW)
    axes[1].plot(positions, attended, 'o-', label='Recibida', color='coral')
    axes[1].plot(positions, attending, 's-', label='Emitida', color='steelblue')
    axes[1].set_xlabel('Posición')
    axes[1].set_ylabel('Suma de pesos')
    axes[1].set_title('Distribución de atención (causal)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Diagonal (auto-atención): {np.trace(attn_matrix) / WINDOW:.4f}")
    print("Triángulo inferior = atención causal (cada token solo ve previos)")

sample_words = words[:WINDOW]
print(f"Primeras {WINDOW} palabras como contexto de prueba:")
print(' '.join(sample_words))
plot_causal_attention(sample_words, model)

In [ ]:
# Sin recompilar — calcula de los logits ya entrenados
best_val_loss = min(history.history['val_loss'])
best_val_ppl = np.exp(best_val_loss)
print(f'Best val perplexity: {best_val_ppl:.2f}')
logits = model.predict(X_test, verbose=0)           # (N, 32, 9204)
last_logits = logits[:, -1, :]                       # última posición
last_probs = tf.nn.softmax(last_logits, axis=-1).numpy()
top5_preds = np.argsort(last_probs, axis=-1)[:, -5:]
y_last = y_test[:, -1]
correct = np.any(top5_preds == y_last[:, None], axis=1)
top5_acc = correct.mean()
print(f'Top-5 accuracy: {top5_acc:.4f}')

## Comparación completa

| Notebook | Modelo | Window | Layers | Vocab | Test Accuracy |
|----------|--------|:-----:|:------:|:-----:|:------------:|
| 18 | RNN vanilla | 5 | 1 RNN | v2 | 0.403 |
| 19 | RNN + gradient clipping | 64 | 1 RNN | v2 | 0.420 |
| 20 | RNN + Bahdanau Attention | 5 | 1 RNN+Att | v2 | **0.575** |
| 21_v3 | Self-Attention manual (trainable emb) | 5 | 1 SA | v2 | **0.749** |
| **22_v2** | **Mini Transformer (causal+last token+trainable emb)** | **5** | **3 SA+FFN** | v2 | **0.755** |
| 23 | Mini Transformer (trainable+3 layers) | 5 | 3 SA+FFN | v3 | **0.177** |
| **23_v2** | **Mini Transformer (Word2Vec v4)** | **5** | **3 SA+FFN** | **v4** | **0.171** |
| **23_v3** | **Mini Transformer (v4 + TF + W32)** | **32** | **3 SA+FFN** | **v4** | **0.292** |
| **23_v4** | **Mini Transformer (v4 + TF + W32 + 6L)** | **32** | **6 SA+FFN (Warmup+LS)** | **v4** | **? (esperado >0.32)** |

**Nota:** 23_v4 sube a 6 capas + label smoothing + warmup. Se espera >0.32 de val_acc.